In [ ]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 3 – Location problems
#  Section: 3.2.2 – The p-median problem on a single node set (PMP-N)
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP            # Modeling language
using HiGHS           # Solver
using CSV             # For reading CSV files
using DataFrames      # For handling DataFrame operations
using Distances       # Distance computations

include("utils/pmp-n_utils.jl")

# Main function to solve p-median problem
function solve_pmp_n(file_path, p)
    # Load latitude and longitude data
    coordinates = CSV.read(file_path, header=true, DataFrame) |> Matrix{Float64}
    
    # Compute Haversine distance matrix
    distance_matrix = Distances.pairwise(Distances.Haversine(), coordinates, dims=1)

    # Number of possible facility / client locations
    n = size(distance_matrix, 1)

    # Create the model
    model = JuMP.Model(HiGHS.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Decision variables
    @variable(model, x[1:n, 1:n], Bin)

    # Objective: minimize total distance
    @objective(model, Min, sum(distance_matrix[i, j] * x[i, j] for i in 1:n, j in 1:n))

    # Constraint: exactly p facilities selected
    @constraint(model, sum(x[j, j] for j in 1:n) == p)

    # Constraint: each client assigned to one facility
    @constraint(model, [i in 1:n], sum(x[i, j] for j in 1:n) == 1)

    # Constraint: assignment only to open facilities
    @constraint(model, [i in 1:n, j in 1:n], x[i, j] <= x[j, j])

    # Solve the model
    JuMP.optimize!(model)

    # Extract solution
    x_value = JuMP.value.(x)
    objective_value = JuMP.objective_value(model)

    # Extract / Print selected facility and assignments
    println("Objective value: $objective_value meters\n")
    assignments = Dict()
    facility_ids = findall(x_value[j, j] > 0.5 for j in 1:n)
    println("Selected Facilities (p=$p): $facility_ids\n")
    assignments = Dict(facility_id => findall(x_value[:, facility_id] .> 0.5) for facility_id in facility_ids)
    for (facility_id, clients) in assignments
        println("Facility: $facility_id | Clients: $clients")
    end

    # Plot the solution
    map = plot_solution(coordinates, assignments, p)
    display(map)
end

# Example usage
solve_pmp_n("data/sjc_coordinates.csv", 3)

PyObject <folium.folium.Map object at 0x0000017ACCAFF3E0>

Objective value: 19211.026718733905 meters

Selected Facilities (p=3): [81, 92, 99]

Facility: 92 | Clients: [1, 6, 10, 23, 25, 29, 30, 33, 40, 42, 45, 52, 54, 55, 68, 74, 79, 80, 82, 90, 91, 92, 94, 95, 101, 102]
Facility: 81 | Clients: [2, 4, 9, 11, 12, 15, 16, 18, 19, 21, 24, 26, 28, 31, 32, 38, 41, 43, 44, 47, 48, 49, 51, 53, 57, 60, 61, 63, 65, 71, 72, 73, 76, 77, 78, 81, 83, 84, 85, 87, 88, 89, 97, 100]
Facility: 99 | Clients: [3, 5, 7, 8, 13, 14, 17, 20, 22, 27, 34, 35, 36, 37, 39, 46, 50, 56, 58, 59, 62, 64, 66, 67, 69, 70, 75, 86, 93, 96, 98, 99]
